# Georeferenced Locality Records and Species Distribution Models for Sub-Saharan African Bats Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema at:
https://sen.science/doi/10.71728/senscience.bfyb-bfjf/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data Loading
We use `mlcroissant` to load metadata and records from the FAIR² dataset.

In [ ]:
# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.bfyb-bfjf/fair2.json'

# Load the dataset object using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Keywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their `@id` values from the dataset schema.

Every entity in the dataset (record sets, fields, columns) can be referenced by its `@id`. The `mlcroissant` library exposes these IDs for robust access.

In [ ]:
# Display all record sets and their @id in the dataset
record_sets = dataset.record_sets
print("Record Sets (@id):")
for rs in record_sets:
    print("    -", rs['@id'], ":", rs.get('name',''))

# For each record set, show available fields and columns by @id
for rs in record_sets:
    print("\nRecord Set:", rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields (@id):")
    for f in fields:
        print("   -", f['@id'])
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("Columns (@id):")
        for c in columns:
            print("   -", c['@id'])

## 3. Data Extraction
Load records from specific record sets into DataFrames for analysis.

Below, we extract some/all record sets. Use the `@id` from above for referencing.

In [ ]:
# Choose record sets to extract (replace with actual @id values from the schema)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for {record_set_id}...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
            print("Fields:", dataframes[record_set_id].columns.tolist())
            print(dataframes[record_set_id].head())
        else:
            print("No records found for this record set.")
    except Exception as e:
        print("Could not load records:", e)

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, categorize records, remove outliers, and group data.

All fields and columns are referenced by their `@id`.

In [ ]:
# Example: Identify the largest tabular record set and analyze numeric fields

selected_record_set_id = None
for rs_id, df in dataframes.items():
    if selected_record_set_id is None or len(df) > len(dataframes[selected_record_set_id]):
        selected_record_set_id = rs_id
    

df = dataframes[selected_record_set_id]
print(f"Analyzing record set: {selected_record_set_id}")

# Find numeric columns (fields with int or float types)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field for analysis: {numeric_field_id}")

    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize numeric distributions and relationships. Use matplotlib for quick plots.

In [ ]:
# Plot histogram for numeric field (if present)
if numeric_fields:
    df[numeric_field_id].plot(kind='hist', bins=20, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If grouping is done, visualize group means
if group_field is not None:
    grouped_df[numeric_field_id].plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² bat locality records and distribution models using the `mlcroissant` library.

- Data was accessed and processed through entity `@id` references.
- Exploratory analysis identified numeric fields, showed filtering and normalization, and enabled group analysis.
- Simple visualizations illustrated data distributions and relationships.

Further downstream tasks could include spatial visualization, machine learning, and more detailed macroecological analysis.